In [1]:
from pytorch_tabnet.tab_model import TabNetClassifier
import xgboost as xgb
from joblib import load

# TabNet 모델 로드
loaded_model = TabNetClassifier()
loaded_model.load_model("/root/Public_Storage/madelab_khw/lg_aimers/code/trained_tabnet_model.zip.zip")

# XGBoost 모델 로드 (Booster 또는 XGBClassifier 중 선택)
model_xgb = xgb.Booster()
model_xgb.load_model("/root/Public_Storage/madelab_khw/lg_aimers/code/xgb_model.model")

/root/anaconda3/envs/khw/lib/python3.8/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")
/root/anaconda3/envs/khw/lib/python3.8/site-packages/pytorch_tabnet/abstract_model.py:454: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where 

In [2]:
import torch
if torch.cuda.is_available() :
    # torch.cuda.device # Context-manager that changes the selected device.
    device = torch.device('cuda:0')
    device_count = torch.cuda.device_count()
    print('GPU is available.')
else :
    device = torch.device('cpu')
    print('GPU is not available.')

print(device)

GPU is available.
cuda:0


In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [4]:
def convert(str):
    return f"'{str}'"

def pre_process(df):
    threshold = 0.8
    df = df.loc[:, df.isnull().mean()<0.8]
    df = df.drop(columns=['ID'])
    
    with open('/root/Public_Storage/madelab_khw/lg_aimers/code/import.txt', 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    lines = map(str.strip, lines)
    lines = map(convert, lines)
    
    for col in df.columns:
        if col not in lines:
            df = df.drop(columns=[col])
    
    for col in df.columns:
        if df[col].dtype == 'object':  # 문자열 컬럼 처리
            df[col].fillna('Unknown', inplace=True)
        else:
            unique_count = df[col].nunique()
            if unique_count <= 4:
                df[col].fillna(0, inplace=True)
            else:
                df[col].fillna(df[col].mean(), inplace=True)
    
    
    return df


In [5]:
def seed_everything(seed = 21):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

In [6]:
import pandas as pd
import os

if os.path.exists('/root/Public_Storage/madelab_khw/lg_aimers/dataset/test.csv'):
    df = pd.read_csv('/root/Public_Storage/madelab_khw/lg_aimers/dataset/test.csv')
else:
    print('none')

pre_df = pre_process(df)
print(pre_df.shape)

(90067, 61)


In [7]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

def pre_tf(pre_df):
    category_columns = pre_df.select_dtypes(include=['object']).columns.tolist()
    numeric_columns = pre_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    
    
    label_encoder = {}
    
    for col in category_columns:
        encoder = LabelEncoder()
        pre_df[col] = encoder.fit_transform(pre_df[col])
    
    X = pre_df[category_columns + numeric_columns]  
   
    return X

pre_df = pre_tf(pre_df)


In [8]:
category_columns = pre_df.select_dtypes(include=['object']).columns.tolist()
numeric_columns = pre_df.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [9]:
import numpy as np
import pandas as pd

X_test_aligned = pd.DataFrame(pre_df.to_numpy(), columns=pre_df.columns).astype(np.float32)

y_pred_proba_tabnet = loaded_model.predict_proba(X_test_aligned.to_numpy())[:, 1]


dtest = xgb.DMatrix(X_test_aligned, feature_names=pre_df.columns.tolist())  # 🚀 Feature Names 설정
y_pred_proba_xgb = model_xgb.predict(dtest)

y_pred_proba_ensemble = (y_pred_proba_xgb + y_pred_proba_tabnet) / 2

test_df = pd.read_csv('/root/Public_Storage/madelab_khw/lg_aimers/dataset/sample_submission.csv')


test_df["probability"] = y_pred_proba_ensemble
test_df.to_csv("five_test_with_predictions.csv", index=False)
print("예측 결과가 test_with_predictions.csv 파일로 저장되었습니다.")

예측 결과가 test_with_predictions.csv 파일로 저장되었습니다.


In [11]:
print(dtest.feature_names)

['시술 시기 코드', '시술 당시 나이', '시술 유형', '특정 시술 유형', '배란 유도 유형', '배아 생성 주요 이유', '총 시술 횟수', '클리닉 내 총 시술 횟수', 'IVF 시술 횟수', 'DI 시술 횟수', '총 임신 횟수', 'IVF 임신 횟수', 'DI 임신 횟수', '총 출산 횟수', 'IVF 출산 횟수', 'DI 출산 횟수', '난자 출처', '정자 출처', '난자 기증자 나이', '정자 기증자 나이', '배란 자극 여부', '단일 배아 이식 여부', '착상 전 유전 진단 사용 여부', '남성 주 불임 원인', '남성 부 불임 원인', '여성 주 불임 원인', '여성 부 불임 원인', '부부 주 불임 원인', '부부 부 불임 원인', '불명확 불임 원인', '불임 원인 - 난관 질환', '불임 원인 - 남성 요인', '불임 원인 - 배란 장애', '불임 원인 - 여성 요인', '불임 원인 - 자궁경부 문제', '불임 원인 - 자궁내막증', '불임 원인 - 정자 농도', '불임 원인 - 정자 면역학적 요인', '불임 원인 - 정자 운동성', '불임 원인 - 정자 형태', '총 생성 배아 수', '미세주입된 난자 수', '미세주입에서 생성된 배아 수', '이식된 배아 수', '미세주입 배아 이식 수', '저장된 배아 수', '미세주입 후 저장된 배아 수', '해동된 배아 수', '해동 난자 수', '수집된 신선 난자 수', '저장된 신선 난자 수', '혼합된 난자 수', '파트너 정자와 혼합된 난자 수', '기증자 정자와 혼합된 난자 수', '동결 배아 사용 여부', '신선 배아 사용 여부', '기증 배아 사용 여부', '대리모 여부', '난자 채취 경과일', '난자 혼합 경과일', '배아 이식 경과일']
